# 🛡️ CyberShield AI v3.1 — Painel + IA estatística no Colab

**Protótipo defensivo e open source.** O painel contém alertas de regras em eventos fictícios e métricas de IA medidas separadamente em conjuntos sintéticos. Sem acesso a serviços reais. Não carregue dados pessoais ou sensíveis.

In [ ]:
!pip -q install ipywidgets pandas matplotlib

In [ ]:
# Motor defensivo — mesmo código do repositório
"""CyberShield AI: defensive, offline-first event analysis."""
from collections import defaultdict
from datetime import datetime, timedelta, timezone
import csv, json, ipaddress
from pathlib import Path

RULES = {
 "CSAI-001": ("Possível força bruta", "T1110", "Validar tentativas com a equipe, revisar autenticação e MFA."),
 "CSAI-002": ("Possível pulverização de senhas", "T1110.003", "Confirmar contas-alvo, limites de autenticação e MFA."),
 "CSAI-003": ("Alteração suspeita de parâmetro OT", "T0836", "Acionar responsável OT; conferir manutenção autorizada e telemetria; NÃO alterar remotamente o equipamento."),
 "CSAI-004": ("Evento de malware informado", "T1204", "Isolar somente com autorização, preservar evidências e abrir investigação."),
}

def read_events(path):
    path=Path(path)
    if path.stat().st_size>5_000_000: raise ValueError("Arquivo maior que 5 MB")
    if path.suffix.lower()=='.csv':
        with path.open(encoding='utf-8-sig',newline='') as f: rows=list(csv.DictReader(f))
    elif path.suffix.lower()=='.json':
        rows=json.loads(path.read_text(encoding='utf-8'))
    else: raise ValueError("Formato suportado: CSV ou JSON")
    if not isinstance(rows,list) or len(rows)>20000: raise ValueError("Esperada lista de até 20.000 eventos")
    return normalize(rows)

def normalize(rows):
    out=[]
    for i,e in enumerate(rows):
        if not isinstance(e,dict): continue
        try:
            dt=datetime.fromisoformat(str(e['timestamp']).replace('Z','+00:00'))
            if dt.tzinfo is None: continue
            dt=dt.astimezone(timezone.utc)
        except (KeyError,ValueError,TypeError): continue
        out.append({"timestamp":dt,"event_type":str(e.get('event_type',''))[:80],
          "src_ip":str(e.get('src_ip',''))[:64],"username":str(e.get('username',''))[:100],
          "asset":str(e.get('asset',''))[:100],"value":str(e.get('value',''))[:100],
          "authorized":str(e.get('authorized','')).strip().lower()})
    return sorted(out,key=lambda e:e['timestamp'])

def alert(rule_id, anchor, evidence, severity, entity):
    name,technique,advice=RULES[rule_id]
    return {"id":f"{rule_id}-{len(evidence):03d}-{anchor.strftime('%Y%m%d%H%M%S')}",
      "rule_id":rule_id,"title":name,"severity":severity,"entity":entity,
      "timestamp":anchor.isoformat(),"evidence_count":len(evidence),
      "evidence":evidence[:10],"mitre_attack":technique,"recommendation":advice,
      "confidence":"heurística: requer validação humana"}

def detect(events,window_minutes=10,fail_threshold=5,spray_accounts=4):
    events=normalize(events) if events and isinstance(events[0].get('timestamp'),str) else events
    result=[]; logins=defaultdict(list)
    for e in events:
        if e['event_type']=='login_failed' and e['src_ip']:logins[e['src_ip']].append(e)
    for ip,rows in logins.items():
        left=0
        for right,e in enumerate(rows):
            while e['timestamp']-rows[left]['timestamp']>timedelta(minutes=window_minutes): left+=1
            w=rows[left:right+1]
            if len(w)>=fail_threshold:
                result.append(alert('CSAI-001',e['timestamp'],[{"timestamp":x['timestamp'].isoformat(),"username":x['username']} for x in w], 'high' if len(w)>=10 else 'medium',ip));break
        left=0
        for right,e in enumerate(rows):
            while e['timestamp']-rows[left]['timestamp']>timedelta(minutes=window_minutes):left+=1
            w=rows[left:right+1]
            if len(set(x['username'] for x in w if x['username']))>=spray_accounts:
                result.append(alert('CSAI-002',e['timestamp'],[{"timestamp":x['timestamp'].isoformat(),"username":x['username']} for x in w], 'high',ip));break
    for e in events:
        if e['event_type']=='ot_setpoint_change' and e['authorized']=='false':
            result.append(alert('CSAI-003',e['timestamp'],[{"asset":e['asset'],"value":e['value']}],'high',e['asset'] or 'OT'))
        if e['event_type']=='malware_detected':
            result.append(alert('CSAI-004',e['timestamp'],[{"asset":e['asset']}],'high',e['asset'] or 'endpoint'))
    return sorted(result,key=lambda x:(x['severity']!='high',x['timestamp']))

def triage(a):
    """Explainable deterministic agent; no external AI or automated remediation."""
    return {"summary":f"{a['title']} associado a {a['entity']}.",
     "assessment":f"{a['evidence_count']} evidência(s); severidade {a['severity']}. Indício, não confirmação.",
     "next_steps":["Preservar logs originais e registrar horários (UTC).",
      a['recommendation'],"Correlacionar com outras fontes antes de classificar como incidente confirmado.",
      "Escalar ao responsável e documentar decisões."],
     "privacy_check":"Verificar se dados pessoais foram afetados; não presumir vazamento ou obrigação automática de comunicar à ANPD.",
     "status":"triagem pendente de revisão humana"}

def export_report(alerts):
    return {"project":"CyberShield AI","version":"1.0.0","data_origin":"Somente dados fornecidos pelo operador",
      "alerts": [{**a,"triage":triage(a)} for a in alerts],"caution":"Ferramenta de apoio, não valida ataque nem substitui equipe SOC/DPO."}


In [ ]:
# Dados sintéticos do projeto (IPs reservados para documentação)
from datetime import datetime, timezone, timedelta
import pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets
base=datetime(2026,11,17,10,tzinfo=timezone.utc)
rows=[]
for i in range(6): rows.append(dict(timestamp=(base+timedelta(minutes=i)).isoformat(),event_type='login_failed',src_ip='192.0.2.10',username='admin'))
for i,name in enumerate(['ana','bruno','carla','daniel']): rows.append(dict(timestamp=(base+timedelta(hours=1,minutes=i)).isoformat(),event_type='login_failed',src_ip='198.51.100.5',username=name))
rows.extend([dict(timestamp=(base+timedelta(hours=2)).isoformat(),event_type='ot_setpoint_change',asset='camara-fria-demo',value='-2',authorized='false'),dict(timestamp=(base+timedelta(hours=2,minutes=1)).isoformat(),event_type='malware_detected',asset='estacao-demo')])
events=normalize(rows)
print(f'Eventos de demonstração: {len(events)}')

## Modelo experimental v2.2 integrado ao painel

O *Isolation Forest* é treinado apenas com dados **sintéticos normais**. O limiar é escolhido no conjunto de calibração e a avaliação é feita em um teste independente. Resultados do motor de regras e modelo estatístico aparecem **separados**. Não utilizar em produção.

In [ ]:
import numpy as np, json, csv
from pathlib import Path
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

rng22 = np.random.default_rng(22011)

def normais22(n, rng):
    nbase = int(n*0.75)
    falhas = np.concatenate([rng.poisson(1.3,nbase),rng.integers(4,9,n-nbase)])
    contas = np.concatenate([rng.integers(1,4,nbase),rng.integers(1,4,n-nbase)])
    ot = np.concatenate([rng.binomial(1,.04,nbase),np.zeros(n-nbase,dtype=int)])
    malware = np.zeros(n,dtype=int)
    x=np.column_stack((falhas,contas,ot,malware))
    return x[rng.permutation(n)]

def suspeitos22(n, rng):
    a=[]
    for i in range(n):
        t=i%4
        if t==0: a.append([int(rng.integers(9,20)),1,0,0])
        elif t==1: a.append([int(rng.integers(8,18)),int(rng.integers(6,14)),0,0])
        elif t==2: a.append([int(rng.integers(0,3)),1,int(rng.integers(3,8)),0])
        else: a.append([int(rng.integers(0,3)),1,0,1])
    return np.asarray(a,dtype=int)

x_train=normais22(700,rng22)
x_cal=np.vstack((normais22(240,rng22),suspeitos22(120,rng22)))
y_cal=np.array([0]*240+[1]*120)
x_test22=np.vstack((normais22(160,rng22),suspeitos22(80,rng22)))
y_test22=np.array([0]*160+[1]*80)
modelo22=IsolationForest(n_estimators=180,max_samples=256,random_state=22,contamination='auto')
modelo22.fit(x_train)
s_cal=modelo22.decision_function(x_cal)
s_test22=modelo22.decision_function(x_test22)

# Não usar dados de teste na seleção de limiar.
candidatos=np.unique(np.quantile(s_cal,np.linspace(0.01,0.99,99)))
selecoes=[]
for limiar in candidatos:
    pred=(s_cal < limiar).astype(int)
    tn,fp,fn,tp=map(int,confusion_matrix(y_cal,pred,labels=[0,1]).ravel())
    fpr=fp/(fp+tn)
    if fpr <= 0.10:
        selecoes.append((f1_score(y_cal,pred,zero_division=0),recall_score(y_cal,pred,zero_division=0),float(limiar)))
if not selecoes:
    raise RuntimeError('Nenhum limiar atende à restrição de FPR na calibração')
_,_,limiar22=max(selecoes,key=lambda t:(t[0],t[1]))
pred22=(s_test22<limiar22).astype(int)
tn22,fp22,fn22,tp22=map(int,confusion_matrix(y_test22,pred22,labels=[0,1]).ravel())
metricas22={
    'aviso':'Experimento sintético; não extrapolar para ambientes reais',
    'treino_normais':len(x_train),'calibracao_normais':240,'calibracao_suspeitos':120,
    'teste_normais':160,'teste_suspeitos':80,'limiar':round(limiar22,5),
    'TP':tp22,'FP':fp22,'TN':tn22,'FN':fn22,
    'precision':round(float(precision_score(y_test22,pred22,zero_division=0)),3),
    'recall':round(float(recall_score(y_test22,pred22,zero_division=0)),3),
    'f1':round(float(f1_score(y_test22,pred22,zero_division=0)),3),
    'fpr':round(float(fp22/(fp22+tn22)),3)
}
assert (tp22+fn22,tn22+fp22)==(80,160)
assert not np.any(np.isnan(s_test22))
Path('cybershield_avaliacao_v22.json').write_text(json.dumps(metricas22,ensure_ascii=False,indent=2),encoding='utf-8')
print('CyberShield AI v2.2 — teste independente:')
print(json.dumps(metricas22,ensure_ascii=False,indent=2))
print('✅ Testes de integridade v2.2 aprovados')

In [ ]:
# Comparação numérica em dados sintéticos, não em eventos reais
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(5,3.8))
cm=np.array([[tn22,fp22],[fn22,tp22]])
ax.imshow(cm,cmap='Blues')
for (i,j),v in np.ndenumerate(cm): ax.text(j,i,str(v),ha='center',va='center',fontsize=14,color=('white' if v>80 else 'black'))
ax.set(xticks=[0,1],yticks=[0,1],xticklabels=['Normal','Suspeito'],yticklabels=['Normal','Suspeito'],xlabel='Previsão do modelo',ylabel='Rótulo simulado',title='Avaliação independente (sintética)')
plt.tight_layout();plt.show()

In [ ]:
# Painel interativo (altere os controles e clique em Analisar)
falhas=widgets.IntSlider(value=5,min=3,max=15,description='Falhas')
contas=widgets.IntSlider(value=4,min=2,max=10,description='Contas')
janela=widgets.IntSlider(value=10,min=1,max=60,description='Minutos')
executar=widgets.Button(description='Analisar eventos',button_style='primary')
saida=widgets.Output()
def analisar(_=None):
    with saida:
        clear_output(wait=True)
        alerts=detect(events,window_minutes=janela.value,fail_threshold=falhas.value,spray_accounts=contas.value)
        print(f'Eventos: {len(events)} | Alertas por regras: {len(alerts)} | Alta: {sum(a["severity"]=="high" for a in alerts)}')
        print(f'IA (teste sintético independente): precisão={metricas22["precision"]:.3f}; recall={metricas22["recall"]:.3f}; FPR={metricas22["fpr"]:.3f}')
        print('Nota: métricas da IA NÃO avaliam estes 12 eventos e não confirmam os alertas acima.')
        for a in alerts:
            t=triage(a)
            print(f'• {a["severity"].upper()} {a["rule_id"]}: {a["title"]} — {a["entity"]}')
            print('  Orientação:',t['next_steps'][1])
        fig,axs=plt.subplots(1,2,figsize=(11,3.5))
        pd.Series([e['event_type'] for e in events]).value_counts().plot.bar(ax=axs[0],title='Eventos por tipo')
        if alerts: pd.Series([a['rule_id'] for a in alerts]).value_counts().plot.bar(ax=axs[1],title='Alertas por regra')
        else: axs[1].text(.1,.5,'Nenhum alerta disparou')
        plt.tight_layout();plt.show()
        with open('cybershield_relatorio_v31.json','w',encoding='utf8') as f: json.dump({'alertas_por_regras':export_report(alerts), 'avaliacao_ml_sintetica':metricas22, 'aviso':'Regras e IA avaliadas em dados distintos; não há validação operacional.'},f,ensure_ascii=False,indent=2)
        print('Relatório: cybershield_relatorio_v31.json')
executar.on_click(analisar)
display(widgets.VBox([falhas,contas,janela,executar,saida]))
analisar()

In [ ]:
# Execute esta célula somente se quiser baixar o relatório
from google.colab import files
files.download('cybershield_relatorio_v31.json')

## Limitações e contribuições

- A avaliação de IA é sintética e **não mede o desempenho sobre os 12 logs exibidos**.
- A matriz de confusão descreve apenas o conjunto de teste simulado.
- Nenhuma decisão de resposta ou bloqueio é automatizada.
- Futuras contribuições: extrair atributos de logs anonimizados, avaliar desempenho em bases autorizadas e integrar o modelo ao fluxo de eventos com metodologia apropriada.

In [ ]:
# Testes básicos de integração
assert metricas22['TP']+metricas22['FN']==80
assert metricas22['TN']+metricas22['FP']==160
assert len(detect(events))==4
assert 0 <= metricas22['fpr'] <= 1
print('✅ Verificações de integração v3.1 aprovadas')